# BIM/CAD Transformer Pipeline — Walkthrough

This notebook walks through a compact transformer pipeline for **object detection, segmentation and classification of CAD/BIM floorplans**, mirroring the kind of automation used to convert BIM drawings into indoor maps.

1. Generate a small synthetic floorplan dataset
2. Train a **ViT** to classify dominant room type
3. Train **DETR-lite** to detect doors, windows, stairs, desks, toilets
4. Train **Mask2Former-lite** to segment walls / floors / doors / windows
5. Visualize predictions side-by-side

All three models are small enough to train on CPU in a few minutes with the default settings.

## 0. Setup
If you're running this for the first time, install requirements:
```bash
pip install -r ../requirements.txt
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.data import ROOM_TYPES, COMPONENT_TYPES, SEG_CLASSES
from src.data.synthetic_floorplan import generate_floorplan, FloorplanConfig
from src.inference.visualize import overlay_segmentation, draw_boxes, make_prediction_figure

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)
print('room types     :', ROOM_TYPES)
print('component types:', COMPONENT_TYPES)
print('seg classes    :', SEG_CLASSES)

## 1. Synthetic CAD/BIM floorplan dataset
A procedural generator produces building plans laid out via binary-space-partitioning and labeled with room types, component bounding boxes, and a semantic segmentation mask. This mirrors the component ontology commonly targeted in BIM automation.

In [ ]:
sample = generate_floorplan(FloorplanConfig(size=256, seed=13))
fig = make_prediction_figure(sample)
plt.show()
print('component counts in this sample:')
for i, c in enumerate(COMPONENT_TYPES):
    n = int((sample['box_labels'] == i).sum())
    print(f'  {c:10s}: {n}')

## 2. Train ViT for room-type classification
A Vision Transformer (Dosovitskiy et al., 2020) classifies the dominant room in each floorplan.
Use `--epochs 10+` for better accuracy; defaults below are tuned for a quick CPU demo.

In [ ]:
from src.data.dataset import FloorplanClsDataset
from src.models.vit import ViTClassifier
from torch.utils.data import DataLoader

train_ds = FloorplanClsDataset(n=120, size=256, seed=0)
val_ds   = FloorplanClsDataset(n=24,  size=256, seed=100_000)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=16)

vit = ViTClassifier(img_size=256, n_classes=len(ROOM_TYPES)).to(device)
print('ViT params:', sum(p.numel() for p in vit.parameters()) / 1e6, 'M')

opt = torch.optim.AdamW(vit.parameters(), lr=3e-4, weight_decay=0.05)
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(3):
    vit.train(); total, n = 0.0, 0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = loss_fn(vit(imgs), labels)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * imgs.size(0); n += imgs.size(0)
    vit.eval(); correct = tot = 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += (vit(imgs).argmax(1) == labels).sum().item()
            tot += labels.size(0)
    print(f'epoch {epoch+1}  loss={total/n:.3f}  val_acc={correct/tot:.3f}')

## 3. Train DETR-lite for component detection
DETR (Carion et al., 2020) frames detection as set prediction. A transformer decoder emits N "object queries", each predicting (class, bounding box). The Hungarian matcher pairs queries with ground-truth boxes for the loss.

In [ ]:
from src.data.dataset import FloorplanDetDataset, collate_detection
from src.models.detr_lite import DETRLite, HungarianMatcher, detr_loss

det_train = FloorplanDetDataset(n=120, size=256, seed=0)
det_val   = FloorplanDetDataset(n=24,  size=256, seed=100_000)
det_train_dl = DataLoader(det_train, batch_size=8, shuffle=True,  collate_fn=collate_detection)
det_val_dl   = DataLoader(det_val,   batch_size=8, shuffle=False, collate_fn=collate_detection)

detr = DETRLite(n_classes=len(COMPONENT_TYPES)).to(device)
matcher = HungarianMatcher()
print('DETR-lite params:', sum(p.numel() for p in detr.parameters()) / 1e6, 'M')

opt = torch.optim.AdamW(detr.parameters(), lr=1e-4, weight_decay=1e-4)
for epoch in range(3):
    detr.train(); total, n = 0.0, 0
    for imgs, tgts in det_train_dl:
        imgs = imgs.to(device)
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]
        out = detr(imgs)
        losses = detr_loss(out, tgts, matcher, n_classes=len(COMPONENT_TYPES))
        opt.zero_grad(); losses['loss'].backward()
        torch.nn.utils.clip_grad_norm_(detr.parameters(), 1.0)
        opt.step()
        total += losses['loss'].item() * imgs.size(0); n += imgs.size(0)
    print(f'epoch {epoch+1}  loss={total/n:.3f}')

## 4. Train Mask2Former-lite for segmentation
Mask2Former (Cheng et al., CVPR 2022) is a unified mask-transformer decoder: each query produces (class, mask). For semantic segmentation we aggregate per-query predictions into per-class mask logits.

In [ ]:
import torch.nn.functional as F
from src.data.dataset import FloorplanSegDataset
from src.models.mask2former_lite import Mask2FormerLite, mask2former_loss

seg_train = FloorplanSegDataset(n=120, size=256, seed=0)
seg_val   = FloorplanSegDataset(n=24,  size=256, seed=100_000)
seg_train_dl = DataLoader(seg_train, batch_size=8, shuffle=True)
seg_val_dl   = DataLoader(seg_val,   batch_size=8, shuffle=False)

m2f = Mask2FormerLite(n_classes=len(SEG_CLASSES)).to(device)
print('Mask2Former-lite params:', sum(p.numel() for p in m2f.parameters()) / 1e6, 'M')
opt = torch.optim.AdamW(m2f.parameters(), lr=2e-4, weight_decay=1e-4)

def mean_iou(logits, target, n_classes):
    pred = F.interpolate(logits, size=target.shape[-2:], mode='bilinear', align_corners=False).argmax(1)
    ious = []
    for c in range(n_classes):
        p, t = (pred == c), (target == c)
        u = (p | t).sum().item()
        if u: ious.append((p & t).sum().item() / u)
    return sum(ious) / max(len(ious), 1)

for epoch in range(3):
    m2f.train(); total, n = 0.0, 0
    for imgs, seg in seg_train_dl:
        imgs, seg = imgs.to(device), seg.to(device)
        out = m2f(imgs)
        losses = mask2former_loss(out, seg, n_classes=len(SEG_CLASSES))
        opt.zero_grad(); losses['loss'].backward()
        torch.nn.utils.clip_grad_norm_(m2f.parameters(), 1.0)
        opt.step()
        total += losses['loss'].item() * imgs.size(0); n += imgs.size(0)
    m2f.eval(); iou_sum = nb = 0
    with torch.no_grad():
        for imgs, seg in seg_val_dl:
            imgs, seg = imgs.to(device), seg.to(device)
            iou_sum += mean_iou(m2f(imgs)['seg_logits'], seg, len(SEG_CLASSES))
            nb += 1
    print(f'epoch {epoch+1}  loss={total/n:.3f}  val_mIoU={iou_sum/nb:.3f}')

## 5. Qualitative results
Run all three models on a handful of unseen floorplans and show ground truth next to predictions.

In [ ]:
from src.data.dataset import FloorplanClsDataset, FloorplanDetDataset, FloorplanSegDataset

N = 4
ds_cls = FloorplanClsDataset(n=N, size=256, seed=200_000)
ds_det = FloorplanDetDataset(n=N, size=256, seed=200_000)
ds_seg = FloorplanSegDataset(n=N, size=256, seed=200_000)

def cxcywh_to_xyxy_px(b, size=256):
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - 0.5*w, cy - 0.5*h, cx + 0.5*w, cy + 0.5*h], -1) * size

fig, axes = plt.subplots(N, 4, figsize=(14, 3.2 * N))
vit.eval(); detr.eval(); m2f.eval()
with torch.no_grad():
    for i in range(N):
        img, cls_gt = ds_cls[i]
        _, det_gt   = ds_det[i]
        _, seg_gt   = ds_seg[i]
        img_np = (img.permute(1,2,0).numpy() * 255).astype(np.uint8)

        pred_cls = int(vit(img.unsqueeze(0).to(device)).argmax(1).item())
        out = detr(img.unsqueeze(0).to(device))
        prob = out['pred_logits'].softmax(-1)[0]
        scores = prob[:,:-1].max(-1).values; labels = prob[:,:-1].argmax(-1)
        keep = scores > 0.5
        boxes_xyxy = cxcywh_to_xyxy_px(out['pred_boxes'][0][keep]).cpu().numpy()
        seg_out = m2f(img.unsqueeze(0).to(device))['seg_logits']
        pred_seg = F.interpolate(seg_out, size=(256,256), mode='bilinear', align_corners=False).argmax(1)[0].cpu().numpy()

        axes[i,0].imshow(img_np); axes[i,0].set_title(f'Input\nGT cls={ROOM_TYPES[cls_gt.item()]}')
        axes[i,1].imshow(overlay_segmentation(img_np, seg_gt.numpy())); axes[i,1].set_title('GT seg')
        axes[i,2].imshow(draw_boxes(img_np, boxes_xyxy, labels[keep].cpu().numpy(), scores=scores[keep].cpu().numpy()))
        axes[i,2].set_title(f'DETR pred  |  ViT={ROOM_TYPES[pred_cls]}')
        axes[i,3].imshow(overlay_segmentation(img_np, pred_seg)); axes[i,3].set_title('Mask2Former pred')
        for a in axes[i]: a.axis('off')
plt.tight_layout(); plt.show()

## 6. Where to go next
- Swap the synthetic generator for **CubiCasa5K** or rasterised **IFC** files.
- Replace the small CNN backbones with pretrained Swin/ConvNeXt from `timm`.
- Post-process segmentation + detection into a **routable wayfinding graph** (corridor skeletonisation + door connectivity).
- Add panoptic Mask2Former supervision (mask + class matching) for instance-level room segmentation.